In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Find project root
def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src" / "config.py").exists():
            return p
    raise FileNotFoundError("Could not find project root")

PROJECT_ROOT = find_project_root(Path.cwd())
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from config import RAW_DATA_DIR, PROCESSED_DATA_DIR

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data: {RAW_DATA_DIR}")
print(f"Processed data: {PROCESSED_DATA_DIR}")

Project root: C:\Users\adebi\OneDrive\Desktop\DataScienceConsultingPortfolio\01_Portfolio-Projects\Lifecycle-Growth-Engine
Raw data: C:\Users\adebi\OneDrive\Desktop\DataScienceConsultingPortfolio\01_Portfolio-Projects\Lifecycle-Growth-Engine\data\raw
Processed data: C:\Users\adebi\OneDrive\Desktop\DataScienceConsultingPortfolio\01_Portfolio-Projects\Lifecycle-Growth-Engine\data\processed


In [2]:
# Load the data
# We use 'ISO-8859-1' encoding because the dataset contains special characters (like £)
df = pd.read_csv(RAW_DATA_DIR / "online_retail_II.csv", encoding='ISO-8859-1')

print(f"Original shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

Original shape: (541910, 8)
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850.0,United Kingdom


In [3]:
# 1. Standardize column names
df.columns = ['invoice', 'stock_code', 'description', 'quantity', 
              'invoice_date', 'price', 'customer_id', 'country']

# 2. Drop missing Customer IDs (we can't track anonymous users)
rows_before = len(df)
df = df.dropna(subset=['customer_id'])
print(f"Dropped {rows_before - len(df)} rows missing Customer ID")

# 3. Convert customer_id to integer
df['customer_id'] = df['customer_id'].astype(int)

# 4. Parse dates
df['invoice_date'] = pd.to_datetime(df['invoice_date'])

# 5. Create a 'total_spend' column
df['total_spend'] = df['quantity'] * df['price']

# 6. Remove negative quantities and zero prices (returns and data errors)
df = df[df['quantity'] > 0]
df = df[df['price'] > 0]

print(f"Cleaned shape: {df.shape}")
print(f"Date range: {df['invoice_date'].min()} to {df['invoice_date'].max()}")
print(f"Unique customers: {df['customer_id'].nunique()}")

Dropped 135080 rows missing Customer ID


C:\Users\adebi\AppData\Local\Temp\ipykernel_9960\1342713558.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['invoice_date'] = pd.to_datetime(df['invoice_date'])


Cleaned shape: (397885, 9)
Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00
Unique customers: 4338


In [4]:
# Let's verify the data looks clean
print("Top 5 countries by sales volume:")
print(df.groupby('country')['total_spend'].sum().sort_values(ascending=False).head(5))

print("\nTotal revenue in dataset: £{:,.2f}".format(df['total_spend'].sum()))

Top 5 countries by sales volume:
country
United Kingdom    7308391.554
Netherlands        285446.340
EIRE               265545.900
Germany            228867.140
France             209042.050
Name: total_spend, dtype: float64

Total revenue in dataset: £8,911,425.90


In [5]:
save_path = PROCESSED_DATA_DIR / "clean_online_retail.csv"
df.to_csv(save_path, index=False)
print(f"Saved cleaned dataset to {save_path}")

Saved cleaned dataset to C:\Users\adebi\OneDrive\Desktop\DataScienceConsultingPortfolio\01_Portfolio-Projects\Lifecycle-Growth-Engine\data\processed\clean_online_retail.csv
